In [1]:
from pathlib import Path
import sys
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from datasets import load_dataset as load_hf_dataset
import numpy as np
import gc
import torch
from datasets import load_dataset
gc.collect()
torch.cuda.empty_cache()

dataset = load_dataset("zacharielegault/PatchCamelyon")
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "MOC" / "utilities").exists()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from MOC.utilities.LogicNet import LogicNet
from MOC.utilities.train_model import train_model
from MOC.utilities.HuggingFace_transform import (
    PatchCamelyonTorchDataset,
    load_hf_dataset,
)
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # LogicNet currently expects 1 channel
    transforms.Resize((28, 28)),                  # LogicNet currently expects 28x28
    transforms.ToTensor(),
])

hf_dataset = load_hf_dataset("zacharielegault/PatchCamelyon")

train_dataset = PatchCamelyonTorchDataset(hf_dataset["train"], transform=transform)
test_dataset = PatchCamelyonTorchDataset(hf_dataset["test"], transform=transform)


/home/maciek/Projects/ConfLGN/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = LogicNet(num_classes=2,dense_num=4,base_dense_dims=[2048,1280,640,640], conv_num=4, kernel_multiplier=4, k=64,tau=28,channels=2)


In [ ]:
model, history = train_model(
        model,
        train_dataset,
        test_dataset,
        lr=2e-1,
        weight_decay=0,
        batch_size=128,
        num_iterations=5000,
        metrics_every=250,
        force_cpu=False,
    )

cuda

iter  250 | train_loss 0.8723 | test_acc_discrete 0.4567 | test_loss_discrete 0.8542 | test_acc_relaxed 0.4846 | test_loss_relaxed 0.8640
iter  500 | train_loss 0.8493 | test_acc_discrete 0.4998 | test_loss_discrete 2.7980 | test_acc_relaxed 0.5154 | test_loss_relaxed 0.7582
iter  750 | train_loss 0.8395 | test_acc_discrete 0.4998 | test_loss_discrete 1.2359 | test_acc_relaxed 0.4533 | test_loss_relaxed 0.8482
iter 1000 | train_loss 0.8161 | test_acc_discrete 0.4998 | test_loss_discrete 3.4531 | test_acc_relaxed 0.5349 | test_loss_relaxed 0.8747
iter 1250 | train_loss 0.8280 | test_acc_discrete 0.4998 | test_loss_discrete 2.7920 | test_acc_relaxed 0.4998 | test_loss_relaxed 0.8042


KeyboardInterrupt: 